# House Prices — Kiểm tra ensemble 80/20 với seed 123

Giữ bộ đặc trưng, tham số và quy tắc lọc như lần seed 42. Chỉ đổi cách chia 5 fold trên cùng tập dữ liệu.

In [1]:
EXPERIMENT_NAME = "15-11-2026_ensemble_seed123_phuc"
RANDOM_STATE = 42
N_SPLITS = 5
MODEL_PARAMS = {
    "n_estimators": 200, "max_depth": 4, "learning_rate": 0.05,
    "random_state": RANDOM_STATE, "n_jobs": 2,
}
AREA_LIMIT = 4000
PRICE_LIMIT = 200000
CV_SEED = 123
LASSO_WEIGHT = 0.2

In [2]:
from itertools import combinations
import json
import sys
import warnings
from datetime import datetime, timedelta, timezone
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline

cwd = Path.cwd().resolve()
project = next(
    path for path in (cwd, *cwd.parents, cwd / "Challenge1_House Prices_ Advanced Regression")
    if (path / "src" / "prepare_data.py").is_file()
    and (path / "data" / "raw" / "train.csv").is_file()
)
sys.path.insert(0, str(project))

try:
    import xgboost
except ModuleNotFoundError as error:
    if error.name != "xgboost":
        raise
    raise ModuleNotFoundError(
        "Kernel chua co XGBoost. Chay %pip install xgboost-cpu roi Restart Kernel."
    ) from error

from src.model import make_xgboost
from src.prepare_data import DATA_DIR, ID, TARGET, extract_features, make_preprocessor
from src.model import make_lasso

In [3]:
train = pd.read_csv(DATA_DIR / "raw" / "train.csv")
X = extract_features(train)
prices = train[TARGET]
assert X.shape[1] == 79
assert np.isfinite(prices).all() and prices.gt(0).all()
y = np.log(prices)

formulas = {
    "TotalSF": "TotalBsmtSF + 1stFlrSF + 2ndFlrSF",
    "TotalBath": "FullBath + BsmtFullBath + 0.5 * (HalfBath + BsmtHalfBath)",
    "AgeAtSale": "YrSold - YearBuilt",
}
X_extra = X.assign(
    TotalSF=X["TotalBsmtSF"] + X["1stFlrSF"] + X["2ndFlrSF"],
    TotalBath=X["FullBath"] + X["BsmtFullBath"] + 0.5 * (X["HalfBath"] + X["BsmtHalfBath"]),
    AgeAtSale=X["YrSold"] - X["YearBuilt"],
)
assert X_extra.shape[1] == 82
display(X_extra[list(formulas)].describe().round(2))

cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=CV_SEED)
splits = list(cv.split(X))
predictions = pd.DataFrame({ID: train[ID], "ln_SalePrice": y, "fold": 0})
for fold, (_, valid_idx) in enumerate(splits, 1):
    predictions.loc[valid_idx, "fold"] = fold
baseline_dir = (project / "experiments" / "14-11-2026_6model_79_phuc"
                / "outputs" / "20261114_224507_500186")
baseline = pd.read_csv(baseline_dir / "validation_predictions.csv")
assert predictions[ID].equals(baseline[ID])
assert not predictions["fold"].equals(baseline["fold"])
assert predictions["fold"].value_counts().to_dict() == dict.fromkeys(range(1, 6), 292)
np.testing.assert_allclose(y, baseline["ln_SalePrice"])
suspect = train["GrLivArea"].gt(AREA_LIMIT) & prices.lt(PRICE_LIMIT)
suspected_outliers = train.loc[suspect, [ID, "GrLivArea", TARGET]].copy()
suspected_outliers["validation_fold"] = predictions.loc[suspect, "fold"]
display(suspected_outliers)

        TotalSF  TotalBath  AgeAtSale
count   1460.00    1460.00    1460.00
mean    2567.05       2.21      36.55
std      821.71       0.79      30.25
min      334.00       1.00       0.00
25%     2009.50       2.00       8.00
50%     2474.00       2.00      35.00
75%     3004.00       2.50      54.00
max    11752.00       6.00     136.00
        Id  GrLivArea  SalePrice  validation_fold
523    524       4676     184750                2
1298  1299       5642     160000                5


In [4]:
xgb_features = X_extra[[*X.columns, "TotalSF", "TotalBath"]]
lasso_params = {"alpha": 0.001, "max_iter": 10000}
rows = []
for fold, (fit_idx, val_idx) in enumerate(splits, 1):
    kept = fit_idx[~suspect.iloc[fit_idx].to_numpy()]
    fold_predictions = {}
    for name, features, estimator in (
        ("XGBoost", xgb_features, make_xgboost(**MODEL_PARAMS)),
        ("Lasso", X_extra, make_lasso(**lasso_params)),
    ):
        pipeline = make_pipeline(make_preprocessor(features), estimator)
        pipeline.fit(features.iloc[kept], y.iloc[kept])
        fold_predictions[name] = pipeline.predict(features.iloc[val_idx]).astype(float)

    fold_predictions["Ensemble 80/20"] = (
        (1 - LASSO_WEIGHT) * fold_predictions["XGBoost"] + LASSO_WEIGHT * fold_predictions["Lasso"]
    )
    row = {"fold": fold}
    for name, values in fold_predictions.items():
        predictions.loc[val_idx, name] = values
        row[name] = root_mean_squared_error(y.iloc[val_idx], values)
    rows.append(row)

scores = pd.DataFrame(rows).set_index("fold")
assert np.isfinite(predictions[scores.columns]).all().all()
summary = pd.DataFrame({"rmse_log_mean": scores.mean(), "rmse_log_std": scores.std(ddof=1)})
summary["delta_vs_xgboost"] = summary["rmse_log_mean"] - summary.loc["XGBoost", "rmse_log_mean"]
summary["folds_improved"] = scores.lt(scores["XGBoost"], axis=0).sum()
display(scores.round(6))
display(summary.round(6))

       XGBoost     Lasso  Ensemble 80/20
fold                                    
1     0.108733  0.101935        0.104179
2     0.144920  0.152138        0.143036
3     0.123420  0.111006        0.117527
4     0.132510  0.120717        0.126770
5     0.138028  0.202098        0.144863
                rmse_log_mean  rmse_log_std  delta_vs_xgboost  folds_improved
XGBoost              0.129522      0.014034          0.000000               0
Lasso                0.137579      0.040730          0.008056               3
Ensemble 80/20       0.127275      0.017223         -0.002247               4


In [5]:
stamp = datetime.now(timezone(timedelta(hours=7))).strftime("%Y%m%d_%H%M%S_%f")
run = project / "experiments" / EXPERIMENT_NAME / "outputs" / stamp
run.mkdir(parents=True, exist_ok=False)
scores.to_csv(run / "validation_by_fold.csv")
summary.to_csv(run / "validation_metrics.csv", index_label="model")
predictions.to_csv(run / "validation_predictions.csv", index=False)
config = {
    "cv_seed": CV_SEED, "folds": N_SPLITS, "xgboost": MODEL_PARAMS,
    "lasso": lasso_params, "lasso_weight": LASSO_WEIGHT,
    "outlier_rule": {"GrLivArea_gt": AREA_LIMIT, "SalePrice_lt": PRICE_LIMIT, "scope": "train only"},
}
(run / "manual_params.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print(run.relative_to(project))

experiments\15-11-2026_ensemble_seed123_phuc\outputs\20261115_121000_620119
